In [ ]:
#file I/O
from pathlib import Path
from typing import List, Tuple
import pandas as pd
from natsort import natsorted

#rdkit imports
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

#AQME Imports
from aqme.csearch import csearch #AQME CSEARCH 
from aqme.qprep import qprep #AQME QPREP
import os #have to use this for qprep

#openbabel - used to convert the aryne SMILES to xyz
from openbabel import pybel, openbabel

"""
### 'Generate_Conformers_Write_DFT_Inputs.ipynb' - @GCH v1.0 - last updt. 05/10/2026

### Notebook Overview:
This notebook contains code to obtain initial 3D geometries/conformers from SMILES data, generate
DFT input files (Orca 5.0.3, M06-2X-D3/def2-SVP, .inp), and batch .inps for submission to an HPC.
You can then submit en masse via a SLURM-based manager (for ex: via pOrca). Output jobs should 
then be processed locally for validation (for ex: via the "Bacon" notebooks in this project). 

### Motivation:
A mostly hands-off means of going from SMILES => Orca DFT input files

### Specifics of the Workflow in this Notebook:
- Initially SMILES are converted to 3D coordinates using CSEARCH (rdkit-based embedding).
- For any valid SMILES string that can't generate a 3D geometry, generally arynes, we use
  OpenBabel/Pybel to generate a desperate attempt for a set of 3D coordinates (see note). 

### A Note on Conformer Generation:
In this phase of the project, hetarynes are generally planar/fused and feature no substituents. 
Accordingly, we can get away with very generic conformer handling (ie, just generate a set of coords
and use those as starting points for DFT optimization). Future versions of this code will have much 
more powerful/robust means of generating conformers and selecting which to use for DFT-input files. 

### Planned Features:
1. Remove CSEARCH Dependency (slow/replace with xTB/GOAT/something else)
2. Remove QPREP Dependency (doens't handle absolute pathlib paths so need to use rel paths/import os; annoying)
3. Improve conformer generation/workflow/variability (in tandem with #1 above)
4. Eventually expand to other job types (vs. just opt+freq / more dynamic vs. a static option only)

### How to Use this Notebook: 
1. Define relevant Paths to output files in the PATH cell below
2. Define DFT Job Parameters
3. Run all cells in the notebook
   
"""

In [ ]:
"""
Define Paths and Relevant Directories in this cell.
"""

#the project TLD is 1 "parent" above current dir (where this notebook is located)
project_tld = Path.cwd().parent

# ### Aryne Dirs ###
Arene_DFT_Dir = project_tld / "DFT_Arene_Data"
Arene_DFT_Dir.mkdir(exist_ok=True, parents=True)

#dir to contain DFT input files 
Arene_DFT_Input_dir = Arene_DFT_Dir / "Arene_Orca_DFT_Inputs"
Arene_DFT_Input_dir.mkdir(exist_ok=True)

#dir to contain pre-optimization .sd-files (conformer data)
Arene_PreOpt_SDF_dir = Arene_DFT_Input_dir / "Arene_PreOpt_SDFs"
Arene_PreOpt_SDF_dir.mkdir(exist_ok=True)

### Aryne Dirs ###
Aryne_DFT_Dir = project_tld / "DFT_Aryne_Data"
Aryne_DFT_Dir.mkdir(exist_ok=True, parents=True)

#dir to contain DFT input files 
Aryne_DFT_Input_dir = Aryne_DFT_Dir / "Aryne_Orca_DFT_Inputs"
Aryne_DFT_Input_dir.mkdir(exist_ok=True)

#dir to contain pre-optimization .sd-files (conformer data)
Aryne_PreOpt_SDF_dir = Aryne_DFT_Input_dir / "Aryne_PreOpt_SDFs"
Aryne_PreOpt_SDF_dir.mkdir(exist_ok=True)

In [ ]:
def validate_dataframe_format(df: pd.DataFrame, required_cols: List[str]) -> bool:
    """
    Function: Validate DataFrame has required columns and basic structure for data import

    Inputs: 
        - df: pd.DataFrame; a Pandas DataFrame to parse for required_cols
        - required_cols: List[str]; a list of column headers required for some task

    Returns:
        - Bool; True = df is OK; False = df not correct
    """

    #check that the passed info is indeed a pd.DF
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame")

    #check the passed df for column headers passed as a list
    missing_cols = [col for col in required_cols if col not in df.columns]

    #if some cols are missing, report to user
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    #if df is completely empty, report to user
    if df.empty:
        raise ValueError("DataFrame is empty")

    #return True if everything OK
    return True

In [ ]:
def get_filepaths_in_target_dir(target_directory: Path, extension: str, printing=True) -> list[Path]:
    """
    Function: returns a list of Path objects pertaining to files in a specified directory. Searches for specific .ext
    
    Input:
        - target_directory; Path containing files with the '.extension'
        - extension; str: a string specifying an extension to use, ex: ".out", ".inp", ".sdf" etc.
        - printing; bool; Default value = True; controls printing back to user
        
    Returns: 
        - filepaths: list[Path]; a list of filepaths for each file wit .extension' in 'target_directory'
    """

    #make sure the incoming is a path
    passed_filepath = Path(target_directory)

    #verify the directory exists as a target dir
    if not passed_filepath.exists() or not passed_filepath.is_dir():
        
        #if not found, print an error (assumes printing is on)
        if printing:
            print(f"Target directory '/{passed_filepath.name}/' not found.")

        #if not found, return an empty list
        return []
    
    #grab user extension with a search wildcard
    file_extension = f"*{extension.lower()}"
    
    #gather and sort .ext files in /target_directory/
    filepaths = target_directory.glob(file_extension)
    filepaths = natsorted(filepaths, key=str)

    #return some user info
    if printing:
        print(f"\033[1mFound {len(filepaths)} {extension} files in '/{passed_filepath.name}/\033[0m'")

    #return the list of located filepaths
    return filepaths

In [ ]:
def check_for_existing_sdfs(target_directory: Path) -> bool:
    """
    Function: Check a target directory for the existence of .sd-files. Returns a Bool to avoid running 
        AQME's CSEARCH repeatedly. 

    Input:
        - target_directory: Path - the target directory to check for existing .sdfs

    Returns:
        - bool 
    """
    
    #make sure the incoming is a path
    passed_filepath = Path(target_directory)

    #First, check if the passed directory exists
    if not passed_filepath.exists() or not passed_filepath.is_dir():
        
        #If not found, report to user
        print(f"Target directory '/passed_filepath.name/' not found or not a directory.")

        #return that the dir was not found => False
        return False

    print(f"Checking '/{passed_filepath.name}/' for existing .sd-files...\n")

    #Now check for .sdfs in the filepath (filepath exists)
    existing_sdfs = get_filepaths_in_target_dir(passed_filepath, ".sdf", printing=False)

    #if this list of paths is populated, it's populated with .sdf paths => they exist
    if existing_sdfs:

        #report that they've been found
        print(f"Found {len(existing_sdfs)} existing .sd-files in '/{passed_filepath.name}/'")

        #return that .sdf's exist!
        return True

    #else, make new .sdfs
    print(f"Target directory '/{passed_filepath.name}/' exists but does not contain .sd-files.\nWriting new .sdfs...")

In [ ]:
def split_dir_into_subdirs(parent_directory_path: Path, num_jobs_in_each_subdir: int):
    """
    Function: Splits files in a target directory into subdirectories containing n jobs each
    
    Input:
        - parent_directory_path: Path; path to the directory containing unsorted DFT input files
        - num_jobs_in_each_subdir: int; int value for the number of jobs desired in each subdir
        
    Returns:
        - N/A; operates on files in dir
    """
    
    #find and sort input files 
    inp_files = get_filepaths_in_target_dir(parent_directory_path, ".inp")
    print(f"Found {len(inp_files)} DFT Orca.inp files in '/{parent_directory_path.name}/'")

    #Assigns 'increment' number of DFT jobs to each subdir
    increment = num_jobs_in_each_subdir
    #keeps track of how many subdirs are created
    subdir_count = 0

    for i in range(0, len(inp_files), increment):

        #grab the filename's header to name a subdir with
        file_path = inp_files[i].stem
        #grab the first word of the filename, in this case either "arene" or "aryne"
        mol_header = str(file_path).split('_', 1)[0]
        
        #formatting the names of the subdirs to be 'arene/aryne_inputs_X-Y' 
        #where X and Y are i and i+increment chunks
        subdir = "{}_{}_{}".format(mol_header+"_inputs", i + 1, i + increment)
    
        #make a subdir named 'job_name_i_(i+increment)' to hold 'increment' # of .inps
        target_subdir = parent_directory_path/subdir
        target_subdir.mkdir(parents=True, exist_ok=True)

        for file in inp_files[i:i+increment]:
            ##move the files to target_subdir
            file.rename(target_subdir/file.name)

In [ ]:
"""
Define ORCA DFT Job Parameters in this cell. 'Orca_Block' is ultimately passed to each job, so check formatting. 

In this project:
LOT: M06-2X(D3) / def2-SVP opt/freqs
Params: Hirshfeld Charges, NBO for NPA, Molecular Orbitals for visualization
"""

#this is the literal route line/keywords in each input file controlling Orca DFT parameters
orca_route_line = r'm062x d3zero def2-SVP def2/J defgrid3 tightSCF Hirshfeld Normalprint Printbasis PrintMOs opt freq'.lstrip()

#this line performs a stability check on the final wavefunction
orca_stability_line = r' %scf Stabperform True end'.lstrip()

#this line saves NBO data
orca_nbo_line = r'%nbo NBOKEYLIST = "$NBO NPA BNDIDX $End" end'.lstrip()

#In this case we just want the main orca line and NBO; all are treated as closed shell
orca_block = f"{orca_route_line}\n\n{orca_nbo_line}\n"

# #debug
# print(orca_block)

In [ ]:
"""
Import Data from .csv  "Arenes_and_Generated_Arynes.csv" made by Module3 located in the Module3 directory.
Checks the imported DF for data/expected column headers. 
"""

#The Module2 directory contains the .csv we need in this notebook
input_csv_dir = project_tld / "Module3_Generate_Arynes_From_Arenes" / "Arenes_and_Generated_Arynes.csv"

#Read in the specified .csv as a pd DataFrame
incoming_data = pd.read_csv(input_csv_dir)

#list containing the required cols
required_cols = ["arene_ID", "arene_smiles",
                "aryne_ID", "aryne_smiles"]

#check that the passed DF is valid for our purposes (has correct cols and data)
validated_df = validate_dataframe_format(incoming_data, required_cols)

#if the DF is ok, proceed and give some info
if validated_df:
    print(f"Found {len(incoming_data)} SMILES strings in target csv file: '{input_csv_dir.name}'\n")

#otherwise, fix the df
else:
    print(f"Passed DataFrame could not be parsed for SMILES or does not contain correct col. headers.\n")

#print some of the df
incoming_data.head()

In [ ]:
"""
This cell prepares an 'Arenes_AQME.csv' specifically formatted for AQME's CSEARCH module. This is a required step
for conformer generation using CSEARCH. 'Arenes_AQME.csv' is written to the Module4 directory. 
"""

### Prep a arenes_aqme.csv file for AQME workflow - ARENES ###
arene_aqme_all = incoming_data.filter(['arene_ID', 'arene_smiles'])

#rename columns to match the specific #needs of AQME's CSEARCH module
arene_aqme_all.rename(columns={'arene_ID': 'code_name', 'arene_smiles': 'SMILES'}, inplace=True) 

#arenes column has repeats since a given aryne may form multiple arynes; need to deduplicate so we don't recalc with dFT
arene_aqme_all.drop_duplicates(subset='SMILES', inplace=True)
arene_aqme_df = arene_aqme_all.reset_index(drop=True)

#Print some diag to user
print(f"Prepared 'Arenes_AQME.csv' containing {len(arene_aqme_df)} Arene SMILES for AQME CSEARCH:\n")

#check that everything is hunky-dory with the df format
print(f"{arene_aqme_df.head(5)}\n")

#Direct to correct name/path for the output file
arenes_aqme_csv_name = Arene_PreOpt_SDF_dir / 'Arenes_AQME.csv'

#write the .csv to target dir
arene_aqme_df.to_csv(arenes_aqme_csv_name, index=False)

#print a summary to user
print(f"Wrote '{arenes_aqme_csv_name.name}' containing {len(arene_aqme_df)} arene SMILES to '/{Arene_PreOpt_SDF_dir.name}/'.")

In [ ]:
"""
This cell prepares an 'Arynes_AQME.csv' specifically formatted for AQME's CSEARCH module. This is a required step
for conformer generation using CSEARCH. 'Arynes_AQME.csv' is written to the Module4 directory. 
"""

### Prep a arynes_aqme.csv file for AQME workflow - ARYNES ###
aryne_aqme_all = incoming_data.filter(['aryne_ID', 'aryne_smiles'])

#rename columns to match the specific #needs of AQME's CSEARCH module
aryne_aqme_all.rename(columns={'aryne_ID': 'code_name', 'aryne_smiles': 'SMILES'}, inplace=True) 

#arenes column has repeats since a given aryne may form multiple arynes; need to deduplicate so we don't recalc with dFT
aryne_aqme_all.drop_duplicates(subset='SMILES', inplace=True)
aryne_aqme_df = aryne_aqme_all.reset_index(drop=True)

#Print some diag to user
print(f"Prepared 'Arynes_AQME.csv' containing {len(aryne_aqme_df)} Aryne SMILES for AQME CSEARCH:\n")

#check that everything is hunky-dory with the df format
print(f"{aryne_aqme_df.head(5)}\n")

#Make a name/path for the output file
arynes_aqme_csv_name = Aryne_PreOpt_SDF_dir / 'Arynes_AQME.csv'

#write the .csv to target dir
aryne_aqme_df.to_csv(arynes_aqme_csv_name, index=False)

#print a summary to user
print(f"Wrote '{arynes_aqme_csv_name.name}' containing {len(aryne_aqme_df)} aryne SMILES to '/{Aryne_PreOpt_SDF_dir.name}/'.")

In [ ]:
"""
This cell uses AQME's CSEARCH module to generate 3D coordinates (written to .sd-files) for ARENES.

The cell outputs new .sd-files containing conformational data to a specified input directory:
    Arene_PreOpt_SDF_dir = Arene_DFT_Input_dir / "Arene_PreOpt_SDFs"

The generated .sd-files will be used by AQME's QPREP to write Orca DFT Input files (.inps). 
"""

#specify the name/location of the "Arenes_AQME.csv"
arenes_aqme_csv_path = Arene_PreOpt_SDF_dir / "Arenes_AQME.csv"

#Check if .sd-files already exist in the target directory (likely already ran CSEARCH)
#skips having to re-run this cell. existing_check is a bool (True => .sdfs exist)
existing_check = check_for_existing_sdfs(Arene_PreOpt_SDF_dir)

#if no .sdfs were found, make new ones
if not existing_check:

    #run AQME's CSEARCH module on "Arenes_AQME.csv"
    #writes .sd-files to /Arene_PreOpt_SDF_dir (slowly)
    csearch(
        #specify the location of the input.csv
        input=arenes_aqme_csv_path,
    
        #use RDKIT to embed confs
        program='rdkit',
    
        #write .sd-files to our pre-optimization dir
        destination=Arene_PreOpt_SDF_dir
        )

In [ ]:
"""
This cell uses AQME's CSEARCH module to generate .xyz conformers (written to .sd-files) for ARYNES.

The cell outputs new .sd-files containing conformational data to a specified input directory:
    Aryne_PreOpt_SDF_dir = Aryne_DFT_Input_dir / "Aryne_PreOpt_SDFs"

The generated .sd-files will be used by AQME's QPREP to write Orca DFT Input files (.inps). 

*Note: RDKIT fails to embed/generate 3D coords. for a lot of the aryne SMILES.
    Since we are using DFT to do full opt/freq validation of our structures, it's OK to have
    some quite bad starting geometries, especially since most of these are planar-ish/fully connected
    and rigid systems. 

    Accordingly, we first use RDKIT/CSEARCH to generate as many "good" starting guesses as we can
    then supplement these with a more laissez faire pybel-based SMILES => SDF conversion (next cell). 
"""

#specify the name/location of the "Arenes_AQME.csv"
arynes_aqme_csv_path = Aryne_PreOpt_SDF_dir / "Arynes_AQME.csv"

#Check if .sd-files already exist in the target directory (likely already ran CSEARCH)
#skips having to re-run this cell. existing_check is a bool (True => .sdfs exist)
existing_check = check_for_existing_sdfs(Aryne_PreOpt_SDF_dir)

#if no .sdfs were found, make new ones
if not existing_check:

    #run AQME's CSEARCH module on "Arenes_AQME.csv"
    #writes .sd-files to /Arene_PreOpt_SDF_dir (slowly)
    csearch(
        #specify the location of the input.csv
        input=arynes_aqme_csv_path,
    
        #use RDKIT to embed confs
        program='rdkit',
    
        #write .sd-files to our pre-optimization dir
        destination=Aryne_PreOpt_SDF_dir
        )

In [ ]:
"""
This cell uses OpenBabel (via Pybel) to convert SMILES => 3D coordinates for any SMILES that 
CSEARCH/RDKIT failed to make an .sdf for. 

The cell outputs new .sd-files containing conformational data to a specified input directory:
    Aryne_PreOpt_SDF_dir = Aryne_DFT_Input_dir / "Aryne_PreOpt_SDFs"

The generated .sd-files will be used by AQME's QPREP to write Orca DFT Input files (.inps). 
"""

#Snag the aryne_ID and aryne_SMILES cols from working df
aryne_df = incoming_data.filter(['aryne_ID', 'aryne_smiles'])

#specify a target dir to write .sd-files to
target_sdf_dir = Aryne_PreOpt_SDF_dir

# Iterate through the aryne dataframe row by row
for index, row in aryne_df.iterrows():

    #specify relevant cols for pybel
    aryne_id = row['aryne_ID']
    aryne_smiles = row['aryne_smiles']

    #Check the /Aryne_PreOpt_SDF_dir directory for an existing .sdf written by CSEARCH
    sd_file = Aryne_PreOpt_SDF_dir / f"{aryne_id}_rdkit.sdf"

    #if no .sdf exists, write one using pybel
    if not sd_file.exists():

        # Create a Molecule object from the SMILES string
        mol = pybel.readstring("smiles", aryne_smiles)
    
        # Add hydrogens and optimize the 3D coordinates
        mol.addh()
        mol.make3D()
    
        # Save the molecule to an SDF file in the output directory
        output_file = target_sdf_dir / f"{aryne_id}_rdkit.sdf"
        mol.write("sdf", str(output_file), overwrite=True)
    
        #debug
        print(f"Saved {aryne_id} to {output_file}")


In [ ]:
"""
This cell generates ORCA DFT input files (.inp) for ARENES using AQME's QPREP Module
QPREP works if you use relative paths
"""

# Store current directory and change to the SDF directory
original_cwd = Path.cwd()
os.chdir(Arene_PreOpt_SDF_dir)

try:
    # Grab all .sdf files in the current directory (now the SDF directory)
    sd_files = list(Path.cwd().glob("*.sdf"))
    
    if not sd_files:
        print("No .sdf files found in the directory")
    else:
        print(f"Found {len(sd_files)} .sdf files")
        
        # Convert to relative path strings (just filenames since we're in the directory)
        sd_files_str = [str(path.name) for path in sd_files]
        
        # Convert destination to absolute path string
        destination_str = str(Arene_DFT_Input_dir.resolve())
        
        # Use QPREP with relative file paths
        qprep(files=sd_files_str,
              program='orca',
              qm_input=orca_block,
              charge=0,
              mult=1,
              mem='16GB',
              nprocs=8,
              destination=destination_str
              )
              
finally:
    # Always return to original directory
    os.chdir(original_cwd)

#remove any of the "extra" conformers - maybe don't want to do this if you don't have planar/non-floppy molecules
inp_files = get_filepaths_in_target_dir(Arene_DFT_Input_dir, ".inp")
num_removed = 0
for input_file in inp_files:
    if not str(input_file.name).endswith('1.inp'):
        input_file.unlink()
        num_removed += 1
        #print(f"File '{input_file}' deleted successfully.") #debug

if num_removed != 0:
    print(f'Extraneous conformers removed: {num_removed}')
elif num_removed == 0:
    print ('No extraneous conformers found.')

prepped_arene_inps = get_filepaths_in_target_dir(Arene_DFT_Input_dir, ".inp")
print(f"\n{len(prepped_arene_inps)} DFT ORCA .inps prepared and written to '/{Arene_DFT_Input_dir.name}/'")

In [ ]:
"""
This cell generates ORCA DFT input files (.inp) for ARYNES using AQME's QPREP Module
QPREP works if you use relative paths
"""

# Store current directory and change to the SDF directory
original_cwd = Path.cwd()
os.chdir(Aryne_PreOpt_SDF_dir)

try:
    # Grab all .sdf files in the current directory (now the SDF directory)
    sd_files = list(Path.cwd().glob("*.sdf"))
    
    if not sd_files:
        print("No .sdf files found in the directory")
    else:
        print(f"Found {len(sd_files)} .sdf files")
        
        # Convert to relative path strings (just filenames since we're in the directory)
        sd_files_str = [str(path.name) for path in sd_files]
        
        # Convert destination to absolute path string
        destination_str = str(Aryne_DFT_Input_dir.resolve())
        
        # Use QPREP with relative file paths
        qprep(files=sd_files_str,
              program='orca',
              qm_input=orca_block,
              charge=0,
              mult=1,
              mem='16GB',
              nprocs=8,
              destination=destination_str
              )
              
finally:
    # Always return to original directory
    os.chdir(original_cwd)

#remove any of the "extra" conformers - maybe don't want to do this if you don't have planar/non-floppy molecules
inp_files = get_filepaths_in_target_dir(Aryne_DFT_Input_dir, ".inp")
num_removed = 0
for input_file in inp_files:
    if not str(input_file.name).endswith('1.inp'):
        input_file.unlink()
        num_removed += 1
        #print(f"File '{input_file}' deleted successfully.") #debug

if num_removed != 0:
    print(f'Extraneous conformers removed: {num_removed}')
elif num_removed == 0:
    print ('No extraneous conformers found.')

prepped_aryne_inps = get_filepaths_in_target_dir(Aryne_DFT_Input_dir, ".inp")
print(f"\n{len(prepped_aryne_inps)} DFT ORCA .inps prepared and written to '/{Aryne_DFT_Input_dir.name}/'")

In [ ]:
"""
This cell splits your targeted directories into subdirectories of n jobs (pass diff. int)
"""

#split jobs into subdirs for submission to HPC 
split_dir_into_subdirs(Arene_DFT_Input_dir, 100) #Arenes

split_dir_into_subdirs(Aryne_DFT_Input_dir, 100) #Arynes

In [ ]:
### Eventually will use something relying on pathlib once QPREP dropped
# """
# This cell generates ORCA DFT input files (.inp) for ARENES using AQME's QPREP Module
# """

# #Grab all aryne.sdf sd-files in the pre-opt .sdf directory
# sd_files = get_filepaths_in_target_dir(Arene_PreOpt_SDF_dir, ".sdf")

# #qprep takes str not path
# sd_files_str = [str(path) for path in sd_files]

# #use AQME's QPREP module to write DFT Input files using our job parameters, specified above
# qprep(files=sd_files_str,
#     program='orca',
#     qm_input=orca_block,
#     charge = 0,
#     mult = 1,
#     mem='16GB',
#     nprocs=8,
#     destination=Arene_DFT_Input_dir
#     )

# #remove any of the "extra" conformers - maybe don't want to do this if you don't have planar/non-floppy molecules
# inp_files = get_filepaths_in_target_dir(Arene_DFT_Input_dir, ".inp")
# num_removed = 0
# for input_file in inp_files:
#     if not str(input_file.name).endswith('1.inp'):
#         input_file.unlink()
#         num_removed += 1
#         #print(f"File '{input_file}' deleted successfully.") #debug

# if num_removed != 0:
#     print(f'Extraneous conformers removed: {num_removed}')
# elif num_removed == 0:
#     print ('No extraneous conformers found.')

# prepped_arene_inps = get_filepaths_in_target_dir(Arene_DFT_Input_dir, ".inp")
# print(f"\n{len(prepped_arene_inps)} DFT ORCA .inps prepared and written to '/{Arene_DFT_Input_dir.name}/'")

In [ ]:
### Eventually will use something relying on pathlib once QPREP dropped
# """
# This cell generates ORCA DFT input files (.inp) for ARYNES using AQME's QPREP Module
# """

# #Grab all aryne.sdf sd-files in the pre-opt .sdf directory
# sd_files = get_filepaths_in_target_dir(Aryne_PreOpt_SDF_dir, ".sdf")

# #qprep takes str not path
# sd_files_str = [str(path) for path in sd_files]

# #use AQME's QPREP module to write DFT Input files using our job parameters, specified above
# qprep(files=sd_files_str,
#     program='orca',
#     qm_input=orca_block,
#     charge = 0,
#     mult = 1,
#     mem='16GB',
#     nprocs=8,
#     destination=Aryne_DFT_Input_dir
#     )

# #remove any of the "extra" conformers - maybe don't want to do this if you don't have planar/non-floppy molecules
# inp_files = get_filepaths_in_target_dir(Aryne_DFT_Input_dir, ".inp")
# num_removed = 0
# for input_file in inp_files:
#     if not str(input_file.name).endswith('1.inp'):
#         input_file.unlink()
#         num_removed += 1
#         #print(f"File '{input_file}' deleted successfully.") #debug

# if num_removed != 0:
#     print(f'Extraneous conformers removed: {num_removed}')
# elif num_removed == 0:
#     print ('No extraneous conformers found.')

# prepped_aryne_inps = get_filepaths_in_target_dir(Aryne_DFT_Input_dir, ".inp")
# print(f"\n{len(prepped_aryne_inps)} DFT ORCA .inps prepared and written to '/{Aryne_DFT_Input_dir.name}/'")